# Fixed-candidate docking pilot

Does filtering, independent scoring or local refinement improve selection from
the same DiffDock-L pool? See `docs/research.md` for the primary-source review.
This notebook contains **no precomputed benchmark result**. Start with development;
apply the unchanged protocol to evaluation. Default: 5 dev + 20 eval complexes,
20 candidates and 3 seeds. Reference coordinates are evaluation-only.

In [ ]:
#@title 1. Get the helper code
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/nassim-arifette/diffdock-geometric-diffusion-docking.git"  #@param {type:"string"}
REPO_REF = "main"  #@param {type:"string"}

IN_COLAB = "google.colab" in sys.modules
here = Path.cwd()
checkout = next((p for p in (here, here.parent) if (p / "docking_tools").is_dir()), None)
if checkout is None:  # fresh Colab VM: fetch the repository that contains docking_tools/
    checkout = Path("/content/diffdock-geometric-diffusion-docking")
    if not checkout.exists():
        subprocess.run(["git", "clone", "--quiet", "--branch", REPO_REF, REPO_URL, str(checkout)], check=True)
    else:  # a checkout from an earlier cell run: pick up newer commits instead of silently using stale code
        for args in (["fetch", "--quiet", "origin", REPO_REF], ["checkout", "--quiet", REPO_REF], ["merge", "--ff-only", "--quiet", "FETCH_HEAD"]):
            if subprocess.run(["git", "-C", str(checkout), *args]).returncode != 0:
                print("warning: could not update", checkout, "- using it as it is")
                break
if str(checkout) not in sys.path:
    sys.path.insert(0, str(checkout))

if IN_COLAB:  # the notebook's own Python only needs these; DiffDock runs in its own pinned environment
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", ['rdkit==2026.9.1', 'py3Dmol==2.5.5', 'pyyaml', 'posebusters==0.6.5']], check=True)

import docking_tools
print("docking_tools", docking_tools.__version__, "from", checkout)

In [ ]:
from pathlib import Path
import json
from docking_tools.drive import ensure_drive, persist_dataset
from docking_tools._util import gpu_info

SPLIT = "dev"  #@param ["dev", "eval"]
GNINA = ""  #@param {type:"string"}
SAVE_TO_DRIVE = True  #@param {type:"boolean"}
WORKDIR = Path("/content/diffdock_work")
DATASET = Path("/content/posebusters_pilot")
OUTPUT = Path("/content/pilot_" + SPLIT)
SEEDS = [0, 1, 2]
SAMPLES = 20
if gpu_info() is None:
    raise RuntimeError("Select a GPU runtime before starting this pilot.")
drive_root = ensure_drive() if SAVE_TO_DRIVE else None
print("Hardware:", gpu_info())
print("Selectors:", "A/B/C/D" if GNINA else "A/B only; C/D unmeasured")

## Freeze inputs

Intersect the original archive with the final 308-case list, apply declared
eligibility checks, and select one ligand per PDB before running any model.
All exclusions and hashes are kept. This does not establish unseen protein families.

In [ ]:
from docking_tools.pilot_dataset import prepare_dataset
from docking_tools.pilot import read_protocol
manifest = DATASET / "pilot_manifest.json"
if not manifest.exists():
    prepare_dataset(DATASET, n_dev=5, n_eval=20, seeds=SEEDS, samples_per_complex=SAMPLES)
protocol, cases = read_protocol(manifest, SPLIT)
if protocol["seeds"] != SEEDS or protocol["samples_per_complex"] != SAMPLES:
    raise ValueError("Frozen protocol settings differ; use a new dataset/output directory.")
print(json.dumps({"selected": [c["complex_id"] for c in cases], "seeds": SEEDS, "samples": SAMPLES}, indent=2))
if drive_root:
    persist_dataset(DATASET, drive_root, "posebusters_pilot")

## Generate once and evaluate

Finished pools are verified and reused. Failures are retained rather than replaced
with easier cases. GNINA is optional: specify its binary to measure C/D, whose
version, hash and commands are saved. The empty value measures only A/B.

In [ ]:
from docking_tools.pilot import run_pilot
state = run_pilot(manifest, workdir=WORKDIR, out=OUTPUT, split=SPLIT,
                  gnina=GNINA or None, drive_root=drive_root)
print({k: state[k] for k in ("status", "n_expected", "n_ok", "n_failed")})

## Paired results

Main outcome: top-1 heavy-atom RMSD **< 2 Å** AND physical validity against the
prepared receptor. No ligand alignment. Failures remain in denominators;
bootstrap uncertainty groups every seed of one complex. This is a pilot.

In [ ]:
from docking_tools.pilot_summary import summarize_pilot
from IPython.display import Markdown, display
summary = summarize_pilot(OUTPUT)
display(Markdown((OUTPUT / "summary.md").read_text()))
if drive_root:
    for filename in ("summary.json", "summary.md"):
        persist_dataset(OUTPUT / filename, drive_root, "pilot_" + SPLIT + "_summary")

## Inspect saved predictions and reference overlays

In [ ]:
from docking_tools.inspector import write_inspector
from IPython.display import FileLink
completed = [c for c in state["cases"].values() if c["status"] == "ok"]
if completed:
    html = write_inspector(OUTPUT / completed[0]["run_dir"])
    display(FileLink(str(html)))
else:
    print("No complete evaluation: inspect pilot_state.json and failure logs.")